<a href="https://colab.research.google.com/github/aditisankara/lucidrt/blob/feat%2Fanalyze-finetune-interpretability/lucidrt_interp_finetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LucidRT Interp — What did the fine-tune change? (base vs LoRA)

**Goal:** localize the **+37% ROUGE-L** gain from the LoRA fine-tune. The base model is frozen; the entire fine-tune is a 20.2 MB adapter (r=8, α=16, on q/k/v/o_proj, 0.14% of params). So the change is *small and localized* — which means it's actually findable.

We answer three questions:
1. **Where** did the weights change most? (per-layer / per-projection ‖ΔW‖)
2. **How** did internal representations shift? (base-vs-FT activation drift, per layer)
3. **When** does the model decide on plain language? (logit lens — does it commit earlier after fine-tuning?)

Runs on a free Colab **T4**. Fill in the `# TODO(you)` lines; hints are right above each one.

## 0. Setup

In [ ]:
# Colab: install the libs. (Skip the pip line if you're on the cluster env.)
# !pip install -q transformers peft accelerate bitsandbytes datasets

import torch, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_DIR = "lucidrt-lora"   # where the fine-tuned adapter lives / will be written
print("device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE=="cuda" else "cpu only")

tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

## 1. Get the adapter (self-healing)

If the adapter folder exists we load it; otherwise we re-run the quick 300-step fine-tune to recreate it. Either way you end up with `ADAPTER_DIR` on disk.

> If your adapter is on Google Drive, mount it and point `ADAPTER_DIR` at it instead of re-training:
> `from google.colab import drive; drive.mount('/content/drive')`

In [ ]:
import os

have_adapter = os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json"))
print("adapter found:" , have_adapter, "at", os.path.abspath(ADAPTER_DIR))

if not have_adapter:
    print("No adapter → running the quick 300-step fine-tune to recreate it...")
    # This mirrors Section 2 of lucidrt_data_pipeline.ipynb. If you'd rather not re-train,
    # stop here, mount Drive, and set ADAPTER_DIR to your saved copy.
    from peft import LoraConfig, get_peft_model
    from datasets import load_dataset

    base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(DEVICE)
    lcfg = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.0, bias="none",
                      target_modules=["q_proj","k_proj","v_proj","o_proj"], task_type="CAUSAL_LM")
    m = get_peft_model(base, lcfg)
    m.print_trainable_parameters()   # expect ~2.18M trainable (0.14%)

    # --- minimal training loop (300 steps) ---------------------------------
    # TODO(you): build the dataset + 300-step loop as in the main notebook, then save.
    #   Hint: load_dataset('lighteval/med_paragraph_simplification'); format each row with the
    #   same prompt template; tokenize; AdamW(lr=2e-4); 300 steps; m.save_pretrained(ADAPTER_DIR)
    raise NotImplementedError("Fill in the 300-step loop, or point ADAPTER_DIR at a saved adapter.")
else:
    print("Using existing adapter — no training needed.")

## 2. Load both models

We keep two models in memory: the **frozen base** and the **fine-tuned** (base + adapter). Everything below compares them. On a T4 two 1.5B fp16 models is ~6 GB — fine.

In [ ]:
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(DEVICE).eval()

# Fine-tuned = base + adapter. merge_and_unload() folds the delta into the weights so it
# behaves like a normal model (handy for activation hooks later).
ft = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16).to(DEVICE)
ft = PeftModel.from_pretrained(ft, ADAPTER_DIR)
ft = ft.merge_and_unload().eval()

print("base and fine-tuned models loaded.")

## 3. Weight-delta ‖BA‖ — *where* did the weights change?

For each adapted module, the update is **ΔW = (α/r)·B·A** (here α/r = 16/8 = **2.0**). A stores as `lora_A.weight` shape `[r, in]`, B as `lora_B.weight` shape `[out, r]`, so `B@A` is `[out, in]` — the same shape as the base weight.

We measure each module's **‖ΔW‖ / ‖W_base‖** (relative Frobenius norm): how big is the nudge compared to the weight it sits on? This is pure linear algebra — no GPU, no forward pass.

> We reload a *fresh* PeftModel here (not merged) so the separate A/B matrices are still accessible.

In [ ]:
from peft import PeftModel
import re, collections

probe = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
probe = PeftModel.from_pretrained(probe, ADAPTER_DIR)

SCALING = 16 / 8   # alpha / r
rows = []
for name, mod in probe.named_modules():
    # a LoRA-wrapped linear has .lora_A and .lora_B submodules
    if hasattr(mod, "lora_A") and hasattr(mod, "lora_B"):
        A = mod.lora_A["default"].weight.data            # [r, in]
        B = mod.lora_B["default"].weight.data            # [out, r]
        Wb = mod.base_layer.weight.data                  # [out, in]
        # TODO(you): compute dW = SCALING * (B @ A), then the relative norm rel = ||dW|| / ||Wb||
        #   Hint: torch.linalg.matrix_norm(X) or X.norm() gives the Frobenius norm.
        dW = None   # <-- TODO
        rel = None  # <-- TODO

        layer = int(re.search(r"layers\.(\d+)\.", name).group(1))
        proj = re.search(r"(q|k|v|o)_proj", name).group(1)
        rows.append((layer, proj, float(dW.norm()), float(rel)))

rows.sort()
print(f"{'layer':>5} {'proj':>5} {'||dW||':>10} {'||dW||/||W||':>13}")
for layer, proj, nrm, rel in rows:
    print(f"{layer:>5} {proj:>5} {nrm:>10.3f} {rel:>13.4f}")

In [ ]:
# Visualize: relative update strength as a layer × projection heatmap.
import matplotlib.pyplot as plt

projs = ["q","k","v","o"]
n_layers = max(r[0] for r in rows) + 1
grid = np.zeros((len(projs), n_layers))
for layer, proj, nrm, rel in rows:
    grid[projs.index(proj), layer] = rel

fig, ax = plt.subplots(figsize=(11, 2.6))
im = ax.imshow(grid, aspect="auto", cmap="viridis")
ax.set_yticks(range(len(projs))); ax.set_yticklabels([p+"_proj" for p in projs])
ax.set_xlabel("layer"); ax.set_title("Relative LoRA update  ||ΔW|| / ||W||")
fig.colorbar(im, ax=ax, shrink=0.8); plt.tight_layout(); plt.show()

# TODO(you): one sentence — which layers / projections carry the biggest change? Early, middle, or late?

## 4. Effective rank of ΔW — is the update concentrated?

The adapter is rank-8 by construction, but the *effective* rank (how many singular values actually matter) can be lower. If ΔW is dominated by 1–2 directions, the fine-tune learned a very specific transform.

In [ ]:
def effective_rank(dW):
    # participation ratio of the singular values: (Σσ)^2 / Σ(σ^2). Ranges 1..r.
    s = torch.linalg.svdvals(dW.float())
    # TODO(you): return (s.sum()**2) / (s**2).sum()  as a float
    return None

# Pick the few highest-||dW|| modules from section 3 and report their effective rank.
# TODO(you): loop over the top-5 rows, recompute dW, print effective_rank(dW).
#   Hint: you already have A, B per module above — refactor the dW computation into a helper
#   so sections 3 and 4 share it.

## 5. Activation drift — *how* did representations shift?

Feed the **same** medical prompts through base and fine-tuned, capture each layer's hidden state, and measure how different they are per layer. A natural metric is **1 − mean cosine similarity** of the last-token hidden state (simple), or linear **CKA** (more robust to rotation). Expect drift to grow through the stack and peak where the fine-tune did its work.

In [ ]:
PROMPT_TEMPLATE = (
    "Simplify the following medical text into plain language a patient can understand.\n\n"
    "Medical text:\n{technical}\n\nPlain language:\n"
)
SAMPLES = [
    "Two trials met the inclusion criteria. Both were small, reported results inadequately, "
    "and were of low methodological quality, giving very-low certainty evidence.",
    "The intervention group showed a statistically significant reduction in systolic blood "
    "pressure relative to placebo at twelve weeks (p < 0.05).",
    "Participants received topical application weekly for six weeks; the primary endpoint was "
    "complete wound epithelialization at follow-up.",
]
prompts = [PROMPT_TEMPLATE.format(technical=t) for t in SAMPLES]

@torch.no_grad()
def layer_hidden_states(model, text):
    enc = tok(text, return_tensors="pt").to(DEVICE)
    out = model(**enc, output_hidden_states=True)
    # out.hidden_states is a tuple (embeddings, layer1, ..., layerN); take the LAST token of each
    return torch.stack([h[0, -1, :].float() for h in out.hidden_states])   # [n_layers+1, d]

# TODO(you): for each prompt, get hidden states from base and ft; per layer compute
#   cosine similarity, then drift = 1 - cosine. Average drift over the prompts.
#   Hint: torch.nn.functional.cosine_similarity(a, b, dim=0)
drift = None   # <-- TODO: array of length n_layers+1

plt.figure(figsize=(9,3))
plt.plot(drift, marker="o")
plt.xlabel("layer (0 = embeddings)"); plt.ylabel("1 − cosine (base vs FT)")
plt.title("Representation drift from fine-tuning, per layer"); plt.tight_layout(); plt.show()

## 6. Logit lens — *when* does the model decide on plain language?

The logit lens projects each layer's residual stream through the final `lm_head` (un-embedding) to ask: *if the model had to answer from this layer, what token would it pick?* If the fine-tune helps the model **commit to plain-language words earlier** in the stack, that's a concrete, visual story.

Pick a position where base and FT diverge (e.g. the first generated token), and track the rank/probability of a plain-language target token across layers for both models.

In [ ]:
@torch.no_grad()
def logit_lens(model, text, target_token_id):
    enc = tok(text, return_tensors="pt").to(DEVICE)
    out = model(**enc, output_hidden_states=True)
    norm = model.model.norm        # Qwen final RMSNorm before lm_head
    head = model.lm_head
    probs_per_layer = []
    for h in out.hidden_states:                      # each [1, seq, d]
        logits = head(norm(h[:, -1, :]))             # project last position
        p = torch.softmax(logits.float(), dim=-1)[0, target_token_id]
        probs_per_layer.append(float(p))
    return probs_per_layer

# TODO(you): choose a plain-language target word the FT model prefers (e.g. ' studies', ' showed'),
#   get its token id with tok(' studies').input_ids[-1], then plot logit_lens() for base vs ft
#   on the same prompt. Does the FT curve rise earlier / higher?
#   Hint: compare against a jargon synonym to make the contrast sharp.

## 7. Synthesis — where does the +37% live?

Tie the three views together in a few sentences (this becomes the interview story / a short section in `results/`):

- **Where (§3–4):** which layers/projections changed most, and how low-rank the change is.
- **How (§5):** where representations diverge — does drift track the high-‖ΔW‖ layers?
- **When (§6):** does the fine-tuned model commit to plain language earlier in the stack?

> *TODO(you): write the 3–4 sentence finding here.*